In [26]:
%pip install -q -U google-genai python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\covin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [27]:
import os
import re
import unicodedata
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types


In [28]:
load_dotenv("api.env")

model_name = "gemini-3.5-flash-lite"

use_mock = False

print("Imported library and established model:", model_name)


Imported library and established model: gemini-3.5-flash-lite


In [ ]:
def load_api_key():
    """Read the Gemini API key from the environment."""
    return os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")


api_key = load_api_key()

if api_key:
    print("API Key found.")
else:
    print("API Key not found. Set use_mock = True to continue in mock mode.")


API Key found.


In [30]:
client = None

if use_mock:
    print("Using Mock Mode - notebook will not call the real API.")
elif not api_key:
    use_mock = True
    print("No API key, automatically switching to Mock Mode.")
else:
    client = genai.Client(api_key=api_key)
    print("Successfully created Gemini Client")


Successfully created Gemini Client


In [31]:
def mock_generate_text(prompt, system_instruction=None):
    """Return a simple response when the real API is unavailable."""
    return (
        "Hello, I am MedBot. This is a sample response in mock mode. "
        "Once a valid API key is provided, this response will be replaced by Gemini."
    )


def generate_text(prompt, system_instruction=None):
    """Call Gemini and fall back to mock mode if the API call fails."""
    if use_mock or client is None:
        return mock_generate_text(prompt, system_instruction)

    config = None
    if system_instruction:
        config = types.GenerateContentConfig(
            system_instruction=system_instruction
        )

    try:
        response = client.models.generate_content(
            model=model_name,
            contents=prompt,
            config=config,
        )
        return response.text
    except Exception as error:
        print("Unable to call the Gemini API. Using a mock response instead.")
        print("Error:", error)
        return mock_generate_text(prompt, system_instruction)


In [32]:
test_prompt = "What should I do for a small scrape?"

answer = generate_text(test_prompt)
print(answer)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


For a small, minor scrape (abrasion), you can usually treat it at home with basic first aid. Here is a simple step-by-step guide:

### 1. Wash Your Hands
Before touching the wound, wash your hands thoroughly with soap and water to avoid introducing bacteria.

### 2. Clean the Wound
*   Rinse the scrape gently with cool or lukewarm running tap water to remove any dirt, gravel, or debris. 
*   Avoid using harsh chemicals like hydrogen peroxide, rubbing alcohol, or iodine, as these can irritate the delicate new skin cells and actually slow down healing.
*   If you see dirt embedded in the scrape that won't wash away, gently clean the area with a washcloth and mild soap. If debris remains deeply embedded, see a healthcare provider.

### 3. Stop Any Bleeding
*   Most small scrapes stop bleeding on their own within a minute or two. 
*   If it is bleeding slightly, apply gentle, direct pressure to the area using a clean cloth or sterile gauze until the bleeding stops.

### 4. Protect and Mois

In [33]:
firstaidkit_path = Path("firstaid_csv")

if firstaidkit_path.exists():
    firstaid_df = pd.read_csv(firstaidkit_path)
else:
    firstaid_df = pd.DataFrame(
        [
            {
                "name": "Adhesive Bandages / Band-Aids",
                "description": "Cover small cuts, scrapes, and blisters to keep out dirt and bacteria."
            },
            {
                "name": "Sterile Gauze Pads",
                "description": "Absorb blood and cover larger cuts or burns."
            },
            {
                "name": "Roller / Crepe Bandages",
                "description": "Wrap sprains, strains, or hold dressings securely in place."
            },
            {
                "name": "Triangular Bandages",
                "description": "Folded to create a sling for broken arms or used as a broad wrap to immobilize injuries."
            },
            {
                "name": "Non-Stick Wound Dressings",
                "description": "Cover raw or oozing wounds without sticking to the healing skin."
            },
            {
                "name": "Medical Tape",
                "description": "Secure gauze and dressings firmly to the skin."
            },
            {
                "name": "Antiseptic wipes or solution",
                "description": "Clean dirt and bacteria from around a wound to prevent infection."
            },
            {
                "name": "Saline solution",
                "description": "Flush dirt, debris, or chemicals out of open wounds and eyes."
            },
            {
                "name": "Antibiotic ointment",
                "description": "Applied to minor cuts to kill bacteria and aid healing."
            },
                        {
                "name": "Disposable non-latex (nitrile) gloves",
                "description": "Protect both the rescuer and the injured person from bodily fluids and contamination."
            },
            {
                "name": "Scissors",
                "description": "Cut tape, gauze, clothing, or bandages safely away from the injury."
            },
            {
                "name": "Tweezers",
                "description": "Remove splinters, small glass shards, or ticks embedded in the skin."
            },
            {
                "name": "CPR face mask/shield",
                "description": "Provide a safe breathing barrier with a one-way valve during rescue breathing or CPR."
            },
            {
                "name": "Instant cold pack",
                "description": "Activated by squeezing to create immediate cold for reducing swelling from sprains, bumps, or insect stings."
            },
            {
                "name": "Emergency thermal blanket",
                "description": "Reflects body heat back to the person to prevent shock or hypothermia in cold environments."
            },
            {
                "name": "Flashlight",
                "description": "Provide visibility during power outages or nighttime emergencies."
            },
            {
                "name": "Notepad and pen",
                "description": "Record the time an injury occurred, symptoms, or medications given for emergency responders."
            }
        ]
    )

firstaid_df

,name,description
0,Adhesive Bandages / Band-Aids,"Cover small cuts, scrapes, and blisters to kee..."
1,Sterile Gauze Pads,Absorb blood and cover larger cuts or burns.
2,Roller / Crepe Bandages,"Wrap sprains, strains, or hold dressings secur..."
3,Triangular Bandages,Folded to create a sling for broken arms or us...
4,Non-Stick Wound Dressings,Cover raw or oozing wounds without sticking to...
5,Medical Tape,Secure gauze and dressings firmly to the skin.
6,Antiseptic wipes or solution,Clean dirt and bacteria from around a wound to...
7,Saline solution,"Flush dirt, debris, or chemicals out of open w..."
8,Antibiotic ointment,Applied to minor cuts to kill bacteria and aid...
9,Disposable non-latex (nitrile) gloves,Protect both the rescuer and the injured perso...


In [34]:
# ==========================================
# MEDICAL KNOWLEDGE DATABASE
# ==========================================

medical_knowledge_data = [
    {
        "category": "wound",
        "situation": "Small cut or scrape",
        "what_to_do": "Rinse gently with clean running water, remove visible dirt gently, and cover with a clean dressing or bandage.",
        "what_not_to_do": "Do not scrub deeply, pick at the wound, or put household chemicals into it.",
        "when_to_seek_help": "Get medical help if bleeding will not stop, the wound is deep or large, something is embedded, or signs of infection develop."
    },

    {
        "category": "bleeding",
        "situation": "Minor bleeding",
        "what_to_do": "Apply steady direct pressure with clean gauze or another clean dressing until bleeding is controlled.",
        "what_not_to_do": "Do not repeatedly lift the dressing to check it.",
        "when_to_seek_help": "Seek urgent medical help for heavy or uncontrolled bleeding."
    },

    {
        "category": "burn",
        "situation": "Minor thermal burn",
        "what_to_do": "Cool the area with cool running water for about 20 minutes. Remove nearby jewelry or clothing that is not stuck to the skin.",
        "what_not_to_do": "Do not use ice directly on the burn or apply butter or oil.",
        "when_to_seek_help": "Get medical help for large or deep burns, burns involving the face, hands, feet, genitals, or major joints, or electrical or chemical burns."
    },

    {
        "category": "sprain_or_strain",
        "situation": "Possible minor sprain or strain",
        "what_to_do": "Stop the activity, protect the area, rest it, and use a wrapped cold pack for short periods to help with pain or swelling.",
        "what_not_to_do": "Do not continue an activity that makes the injury worse.",
        "when_to_seek_help": "Get medical advice if pain or swelling is severe, the limb cannot be used normally, or a fracture may be possible."
    },

    {
        "category": "fracture_or_injury",
        "situation": "Possible broken bone",
        "what_to_do": "Keep the injured area as still and comfortable as possible and get professional medical assessment.",
        "what_not_to_do": "Do not try to straighten a deformed limb or push a protruding bone back into place.",
        "when_to_seek_help": "Seek urgent medical help, especially for severe pain, deformity, an open wound, numbness, or poor circulation."
    },

    {
        "category": "nosebleed",
        "situation": "Nosebleed",
        "what_to_do": "Sit upright, lean slightly forward, and pinch the soft part of the nose continuously for about 10 to 15 minutes.",
        "what_not_to_do": "Do not tilt the head backward.",
        "when_to_seek_help": "Get medical help if bleeding is heavy, follows a significant injury, causes weakness or fainting, or does not stop."
    },

    {
        "category": "eye_problem",
        "situation": "Dust or small irritant in the eye",
        "what_to_do": "Do not rub the eye. Rinse gently with clean water or sterile saline if available.",
        "what_not_to_do": "Do not rub the eye or try to remove an object that appears embedded.",
        "when_to_seek_help": "Get urgent help for vision loss, severe pain, chemical exposure, or an embedded object."
    },

    {
        "category": "chemical_exposure",
        "situation": "Chemical exposure to the eye",
        "what_to_do": "Immediately rinse the eye continuously with clean running water and seek urgent professional medical help.",
        "what_not_to_do": "Do not rub the eye or put other substances into it unless instructed by a professional.",
        "when_to_seek_help": "Treat significant chemical eye exposure as urgent and seek professional help immediately."
    },

    {
        "category": "bite_or_sting",
        "situation": "Minor insect bite or sting",
        "what_to_do": "Wash the area and use a wrapped cold pack to help with swelling or discomfort.",
        "what_not_to_do": "Do not scratch the area excessively.",
        "when_to_seek_help": "Seek urgent help for trouble breathing, swelling of the tongue or throat, fainting, or other signs of a severe allergic reaction."
    },

    {
        "category": "animal_bite",
        "situation": "Animal or human bite",
        "what_to_do": "Wash the wound thoroughly with soap and running water and cover it with a clean dressing.",
        "what_not_to_do": "Do not ignore a bite just because the wound looks small.",
        "when_to_seek_help": "Get medical advice promptly because bites can become infected and may require additional assessment."
    },

    {
        "category": "splinter",
        "situation": "Small superficial splinter",
        "what_to_do": "Wash the area and, if the splinter is small and easy to grasp, remove it gently with clean tweezers. Clean and cover the area afterward.",
        "what_not_to_do": "Do not dig deeply into the skin or force out a deeply embedded object.",
        "when_to_seek_help": "Get medical help if the object is deeply embedded, difficult to remove, or the area becomes increasingly painful, red, swollen, or produces pus."
    },

    {
        "category": "blister",
        "situation": "Small friction blister",
        "what_to_do": "Protect the blister from further rubbing and keep it clean and covered.",
        "what_not_to_do": "Do not deliberately pop a blister unless a healthcare professional advises it.",
        "when_to_seek_help": "Seek medical advice if it becomes increasingly red, painful, swollen, or produces pus."
    },

    {
        "category": "heat_illness",
        "situation": "Heat exhaustion symptoms",
        "what_to_do": "Move to a cool place, rest, loosen unnecessary clothing, and drink fluids if the person is awake and able to swallow safely.",
        "what_not_to_do": "Do not leave the person alone if they become confused or seriously unwell.",
        "when_to_seek_help": "Confusion, loss of consciousness, seizures, or severe deterioration require urgent professional help."
    },

    {
        "category": "hypothermia",
        "situation": "Mild cold exposure",
        "what_to_do": "Move to a warmer place, remove wet clothing, and warm the person gradually with dry layers or a thermal blanket.",
        "what_not_to_do": "Do not use very hot water or intense direct heat.",
        "when_to_seek_help": "Severe shivering, confusion, unusual drowsiness, loss of coordination, or unconsciousness requires urgent medical help."
    },

    {
        "category": "fainting",
        "situation": "Person who has fainted and recovered",
        "what_to_do": "Help them rest safely and check that they are responsive and breathing normally. Let them recover gradually.",
        "what_not_to_do": "Do not give food or drink while the person is not fully alert.",
        "when_to_seek_help": "Urgent help is needed if they do not quickly recover, have trouble breathing, chest pain, a serious injury, or repeated fainting."
    },

    {
        "category": "seizure",
        "situation": "Person having a seizure",
        "what_to_do": "Protect the person from nearby hazards, cushion the head if possible, and stay with them. When the seizure ends, check breathing and recovery.",
        "what_not_to_do": "Do not restrain the person or put objects or fingers in their mouth.",
        "when_to_seek_help": "Seek emergency help for a first seizure, repeated seizures, serious injury, breathing problems, or failure to recover normally."
    },

    {
        "category": "choking",
        "situation": "Possible choking",
        "what_to_do": "If the person cannot cough, speak, or breathe normally, follow recognized choking first-aid procedures and contact emergency services when indicated.",
        "what_not_to_do": "Do not blindly sweep inside the mouth for an object.",
        "when_to_seek_help": "Severe choking is an emergency and requires immediate professional assistance."
    },

    {
        "category": "drowning",
        "situation": "Person rescued from water",
        "what_to_do": "Get the person to safety without putting yourself in danger and check responsiveness and breathing. Contact emergency services when appropriate.",
        "what_not_to_do": "Do not put yourself at risk by entering unsafe water.",
        "when_to_seek_help": "Any significant drowning or breathing problem after water exposure needs urgent professional assessment."
    },

    {
        "category": "poisoning",
        "situation": "Possible poisoning or harmful substance exposure",
        "what_to_do": "Move away from the exposure if safe. Keep the product or container information available and contact local poison-control or emergency services as appropriate.",
        "what_not_to_do": "Do not make the person vomit unless specifically instructed by a poison-control professional.",
        "when_to_seek_help": "Get urgent help for breathing problems, unconsciousness, seizures, severe symptoms, or a potentially dangerous exposure."
    },

    {
        "category": "head_injury",
        "situation": "Minor head bump without emergency warning signs",
        "what_to_do": "Stop the activity, rest, monitor for worsening or new symptoms, and arrange medical advice when needed.",
        "what_not_to_do": "Do not return immediately to sports or activities if symptoms are present.",
        "when_to_seek_help": "Seek urgent help for loss of consciousness, repeated vomiting, seizure, worsening severe headache, confusion, unusual behavior, or weakness."
    },

    {
        "category": "wound_infection",
        "situation": "Possible wound infection",
        "what_to_do": "Keep the wound clean and covered and monitor changes.",
        "what_not_to_do": "Do not ignore worsening redness, swelling, pain, warmth, or drainage.",
        "when_to_seek_help": "Seek medical advice if symptoms are worsening or fever or other signs of illness develop."
    },

    {
        "category": "minor_pain",
        "situation": "Minor pain after a small injury",
        "what_to_do": "Stop the activity, protect the area, and use simple first-aid measures appropriate to the injury, such as a wrapped cold pack for swelling.",
        "what_not_to_do": "Do not keep using an injured body part if activity clearly increases pain.",
        "when_to_seek_help": "Get medical advice for severe, persistent, or worsening pain or pain with other concerning symptoms."
    },

    {
        "category": "recovery",
        "situation": "Monitoring after a minor injury",
        "what_to_do": "Keep the area clean, watch for changes, and follow instructions provided by a healthcare professional.",
        "what_not_to_do": "Do not assume worsening symptoms are normal because the original injury seemed minor.",
        "when_to_seek_help": "Seek medical advice if symptoms worsen, new concerning symptoms appear, or recovery is not progressing as expected."
    }
]

# Convert the list into a DataFrame
medical_knowledge_df = pd.DataFrame(medical_knowledge_data)

print("Medical knowledge database loaded!")
print("Number of entries:", len(medical_knowledge_df))

medical_knowledge_df.head()

Medical knowledge database loaded!
Number of entries: 23


,category,situation,what_to_do,what_not_to_do,when_to_seek_help
0,wound,Small cut or scrape,"Rinse gently with clean running water, remove ...","Do not scrub deeply, pick at the wound, or put...","Get medical help if bleeding will not stop, th..."
1,bleeding,Minor bleeding,Apply steady direct pressure with clean gauze ...,Do not repeatedly lift the dressing to check it.,Seek urgent medical help for heavy or uncontro...
2,burn,Minor thermal burn,Cool the area with cool running water for abou...,Do not use ice directly on the burn or apply b...,"Get medical help for large or deep burns, burn..."
3,sprain_or_strain,Possible minor sprain or strain,"Stop the activity, protect the area, rest it, ...",Do not continue an activity that makes the inj...,Get medical advice if pain or swelling is seve...
4,fracture_or_injury,Possible broken bone,Keep the injured area as still and comfortable...,Do not try to straighten a deformed limb or pu...,"Seek urgent medical help, especially for sever..."


In [35]:
# First-aid search

STOPWORDS = {
    "i", "me", "my", "a", "an", "the", "to", "do", "what", "can",
    "should", "if", "have", "has", "got", "get", "for", "and", "or",
    "is", "it", "of", "on", "in", "with", "this", "that", "use",
    "using", "please", "help", "some", "very", "little", "bit"
}


def normalize_text(text):
    """Lowercase and normalize text for simple keyword matching."""
    text = str(text).casefold()
    return unicodedata.normalize("NFKC", text)


def search_first_aid_kit(query, top_n=5):
    """Find kit items using meaningful whole-word matches."""
    query_words = {
        word for word in re.findall(r"\b[\w'-]+\b", normalize_text(query))
        if word not in STOPWORDS and len(word) > 1
    }

    results = []

    for _, row in firstaid_df.iterrows():
        name = normalize_text(row["name"])
        description = normalize_text(row["description"])
        text_words = set(re.findall(r"\b[\w'-]+\b", f"{name} {description}"))

        score = len(query_words & text_words)

        if score > 0:
            results.append((score, row["name"], row["description"]))

    results.sort(key=lambda item: (-item[0], item[1]))
    return results[:top_n]


# Test
results = search_first_aid_kit("I have a small cut")

for score, name, description in results:
    print(f"{name}: {description}")


Adhesive Bandages / Band-Aids: Cover small cuts, scrapes, and blisters to keep out dirt and bacteria.
Scissors: Cut tape, gauze, clothing, or bandages safely away from the injury.
Tweezers: Remove splinters, small glass shards, or ticks embedded in the skin.


In [36]:
#Medical knowledge search

def search_medical_knowledge(query, top_n=5):
    """Search the built-in medical knowledge database."""

    query_words = {
        word
        for word in re.findall(
            r"\b[\w'-]+\b",
            normalize_text(query)
        )
        if word not in STOPWORDS and len(word) > 1
    }

    results = []

    for _, row in medical_knowledge_df.iterrows():

        searchable_text = " ".join([
            str(row["category"]),
            str(row["situation"]),
            str(row["what_to_do"]),
            str(row["what_not_to_do"]),
            str(row["when_to_seek_help"])
        ])

        text_words = set(
            re.findall(
                r"\b[\w'-]+\b",
                normalize_text(searchable_text)
            )
        )

        score = len(query_words & text_words)

        if score > 0:
            results.append({
                "score": score,
                "category": row["category"],
                "situation": row["situation"],
                "what_to_do": row["what_to_do"],
                "what_not_to_do": row["what_not_to_do"],
                "when_to_seek_help": row["when_to_seek_help"]
            })

    results.sort(
        key=lambda item: (-item["score"], item["category"])
    )

    return results[:top_n]

In [37]:
# First-aid processing

def get_kit_context(question):
    results = search_first_aid_kit(question)

    if not results:
        return "No relevant first-aid kit items were found."

    context = "Relevant items available in the first-aid kit:\n\n"

    for score, name, description in results:
        context += f"- {name}: {description}\n"

    return context


# Test
question = "What can I use if I have a scrape?"
print(get_kit_context(question))


No relevant first-aid kit items were found.


In [38]:
#Medical knowledge processing

def get_medical_knowledge_context(question, top_n=5):

    results = search_medical_knowledge(
        question,
        top_n=top_n
    )

    if not results:
        return "No relevant medical knowledge was found."

    context = "RELEVANT MEDICAL KNOWLEDGE:\n\n"

    for item in results:
        context += (
            f"Category: {item['category']}\n"
            f"Situation: {item['situation']}\n"
            f"What to do: {item['what_to_do']}\n"
            f"What not to do: {item['what_not_to_do']}\n"
            f"When to seek help: {item['when_to_seek_help']}\n\n"
        )

    return context

In [39]:
def build_data_context(menu_df):
    """Convert the first-aid dataframe into readable text."""
    lines = []

    for _, row in menu_df.iterrows():
        name = "" if pd.isna(row.get("name", "")) else str(row.get("name", "")).strip()
        description = "" if pd.isna(row.get("description", "")) else str(row.get("description", "")).strip()

        if name or description:
            lines.append(f"- {name}: {description}")

    return "\n".join(lines)


data_context = build_data_context(firstaid_df)
print(data_context[:800])


- Adhesive Bandages / Band-Aids: Cover small cuts, scrapes, and blisters to keep out dirt and bacteria.
- Sterile Gauze Pads: Absorb blood and cover larger cuts or burns.
- Roller / Crepe Bandages: Wrap sprains, strains, or hold dressings securely in place.
- Triangular Bandages: Folded to create a sling for broken arms or used as a broad wrap to immobilize injuries.
- Non-Stick Wound Dressings: Cover raw or oozing wounds without sticking to the healing skin.
- Medical Tape: Secure gauze and dressings firmly to the skin.
- Antiseptic wipes or solution: Clean dirt and bacteria from around a wound to prevent infection.
- Saline solution: Flush dirt, debris, or chemicals out of open wounds and eyes.
- Antibiotic ointment: Applied to minor cuts to kill bacteria and aid healing.
- Disposable no


In [40]:
# Emergency check

# Fast backup list for common English emergency phrases.
# Gemini is the multilingual semantic layer.

EMERGENCY_TERMS = [
    "can't breathe",
    'cannot breathe',
    'cannot breath',
    "can't breath",
    'difficulty breathing',
    'trouble breathing',
    'having trouble breathing',
    'hard to breathe',
    'struggling to breathe',
    'struggling for air',
    'shortness of breath',
    'severe shortness of breath',
    'breathing problem',
    'breathing problems',
    'not breathing',
    'stopped breathing',
    'stops breathing',
    'barely breathing',
    'gasping for air',
    'gasping',
    'gasping breaths',
    'unable to breathe',
    'unable to catch my breath',
    "can't catch my breath",
    'airway blocked',
    'blocked airway',
    'airway obstruction',
    'choking',
    'choking on',
    'choked on',
    'something stuck in throat',
    'food stuck in throat',
    'object stuck in throat',
    'throat blocked',
    "can't swallow and breathe",
    'turning blue',
    'lips turning blue',
    'blue lips',
    'blue around the lips',
    'severe chest pain',
    'chest pain',
    'chest pressure',
    'pressure in chest',
    'tightness in chest',
    'crushing chest pain',
    'heavy chest',
    'pain in chest',
    'pain spreading to arm',
    'chest pain spreading to arm',
    'chest pain spreading to jaw',
    'chest pain spreading to back',
    'pain in left arm',
    'sudden chest pain',
    'heart attack',
    'possible heart attack',
    'heart problem',
    'heart stopped',
    'heart is racing',
    'heart racing with chest pain',
    'irregular heartbeat with fainting',
    'severe palpitations',
    'palpitations with chest pain',
    'fainted with chest pain',
    'unconscious',
    'unresponsive',
    'not responding',
    "won't respond",
    "doesn't respond",
    'not waking up',
    "can't wake up",
    'cannot wake up',
    'passed out',
    'fainted',
    'fainting',
    'lost consciousness',
    'loss of consciousness',
    'collapsed',
    'suddenly collapsed',
    'collapsed and not responding',
    'not conscious',
    'barely conscious',
    'losing consciousness',
    'about to pass out',
    'seizure',
    'seizures',
    'having a seizure',
    'having seizures',
    'convulsion',
    'convulsions',
    'convulsing',
    'seizing',
    'epileptic seizure',
    "seizure won't stop",
    'repeated seizures',
    'multiple seizures',
    'first seizure',
    'unresponsive after seizure',
    'not waking after seizure',
    'stroke',
    'possible stroke',
    'signs of stroke',
    'sudden weakness',
    'sudden numbness',
    'sudden confusion',
    'suddenly confused',
    "can't speak",
    'cannot speak',
    'sudden trouble speaking',
    'slurred speech',
    'speech suddenly slurred',
    'face drooping',
    'facial drooping',
    'one side of face drooping',
    'arm suddenly weak',
    'leg suddenly weak',
    'one-sided weakness',
    'weakness on one side',
    'numbness on one side',
    'sudden vision loss',
    'sudden blurry vision',
    'sudden severe dizziness',
    'sudden loss of balance',
    'sudden difficulty walking',
    'sudden severe headache',
    'worst headache',
    'sudden severe headache with confusion',
    'severe bleeding',
    'heavy bleeding',
    'uncontrolled bleeding',
    "bleeding won't stop",
    "bleeding won't slow down",
    "blood won't stop",
    "can't stop bleeding",
    'cannot stop bleeding',
    'bleeding heavily',
    'bleeding a lot',
    'large amount of blood',
    'blood loss',
    'major blood loss',
    'spurting blood',
    'blood is spurting',
    'blood spraying',
    'severe blood loss',
    'deep wound with heavy bleeding',
    'deep cut with heavy bleeding',
    'major injury',
    'serious injury',
    'severe injury',
    'major trauma',
    'severe trauma',
    'bad accident',
    'serious accident',
    'car accident',
    'motorcycle accident',
    'vehicle accident',
    'hit by a car',
    'hit by vehicle',
    'pedestrian accident',
    'crush injury',
    'crushed',
    'trapped',
    'person trapped',
    'severe fall',
    'fell from height',
    'fall from a height',
    'head injury',
    'serious head injury',
    'head trauma',
    'severe head injury',
    'neck injury',
    'spinal injury',
    'back injury after accident',
    'possible spinal injury',
    'multiple injuries',
    'severe head injury',
    'loss of consciousness after head injury',
    'unconscious after hitting head',
    'confused after hitting head',
    "can't stay awake after head injury",
    'repeated vomiting after head injury',
    'seizure after head injury',
    'sudden confusion',
    'sudden inability to speak',
    'sudden inability to move',
    'sudden weakness',
    'sudden numbness',
    'sudden severe headache',
    'sudden neurological symptoms',
    'severe burn',
    'serious burn',
    'major burn',
    'large burn',
    'deep burn',
    'extensive burn',
    'burn over a large area',
    'burned over a large area',
    'burn to face',
    'burn to eyes',
    'burn to airway',
    'burned airway',
    'inhaled smoke',
    'smoke inhalation',
    'difficulty breathing after smoke',
    'burn from electricity',
    'electrical burn',
    'electrical injury',
    'electric shock',
    'severe electric shock',
    'chemical burn',
    'chemical exposure to eyes',
    'chemical in eye',
    'chemical exposure',
    'anaphylaxis',
    'anaphylactic reaction',
    'severe allergic reaction',
    'serious allergic reaction',
    'allergic reaction and trouble breathing',
    'allergic reaction with breathing problems',
    'throat swelling',
    'throat closing',
    'throat feels closed',
    'tongue swelling',
    'tongue is swelling',
    'swollen tongue',
    'face swelling with breathing problems',
    'lips swelling with breathing problems',
    'difficulty breathing after eating',
    'difficulty breathing after sting',
    'difficulty breathing after bee sting',
    'difficulty breathing after insect bite',
    'poisoning',
    'poisoned',
    'possible poisoning',
    'toxic exposure',
    'chemical poisoning',
    'swallowed poison',
    'swallowed a chemical',
    'drank a chemical',
    'chemical ingestion',
    'dangerous substance swallowed',
    'unknown substance swallowed',
    'fumes causing breathing problems',
    'toxic fumes',
    'gas exposure',
    'carbon monoxide exposure',
    'carbon monoxide poisoning',
    'overdose',
    'possible overdose',
    'suspected overdose',
    'drug poisoning',
    'drowning',
    'drowning emergency',
    'nearly drowned',
    'pulled from water',
    'found unconscious in water',
    'not breathing after drowning',
    'breathing problems after drowning',
    'underwater too long',
    'water inhalation',
    'almost drowned',
    'severe abdominal pain',
    'severe stomach pain',
    'sudden severe stomach pain',
    'severe belly pain',
    'severe abdominal pain with fainting',
    'severe abdominal pain with vomiting blood',
    'vomiting blood',
    'throwing up blood',
    'blood in vomit',
    'black vomit',
    'bloody stool',
    'blood in stool with weakness',
    'severe internal bleeding',
    'possible internal bleeding',
    "can't keep fluids down",
    'cannot keep fluids down',
    'severe vomiting',
    'vomiting continuously',
    'vomiting nonstop',
    'repeated vomiting with confusion',
    'repeated vomiting with fainting',
    'severe dehydration',
    'extremely dehydrated',
    'confused from dehydration',
    'fainted from dehydration',
    'chemical in eye',
    'chemical in my eye',
    'lost vision',
    'sudden vision loss',
    "suddenly can't see",
    'suddenly cannot see',
    'severe eye injury',
    'serious eye injury',
    'object embedded in eye',
    'something stuck in eye',
    'eye puncture',
    'puncture to eye',
    'open fracture',
    'compound fracture',
    'bone sticking out',
    'bone exposed',
    'severe deformity',
    'limb looks deformed',
    'arm looks deformed',
    'leg looks deformed',
    'severe crush injury',
    'amputation',
    'severed finger',
    'severed hand',
    'severed foot',
    'severed limb',
    'heat stroke',
    'heatstroke',
    'severe heat illness',
    'extreme overheating',
    'confused from heat',
    'unconscious from heat',
    'seizure from heat',
    'severe hypothermia',
    'extremely cold',
    'unconscious from cold',
    'confused from cold',
    'pregnancy emergency',
    'severe bleeding during pregnancy',
    'heavy bleeding during pregnancy',
    'severe abdominal pain during pregnancy',
    'severe abdominal pain while pregnant',
    'fainted while pregnant',
    'unconscious while pregnant',
    'life threatening',
    'life-threatening',
    'life threatening situation',
    'medical emergency',
    'medical emergency right now',
    'need an ambulance',
    'call an ambulance',
    'should I call an ambulance',
    'do I need an ambulance',
    'need emergency help',
    'need emergency medical help',
    'emergency services',
    'call emergency services',
    'get emergency help',
    'urgent medical help',
    'urgent medical attention',
    'immediate medical attention',
    'immediate medical help',
    'right now',
    'happening right now',
    'getting worse quickly',
    'suddenly getting worse'
]


def english_emergency_match(question):
    """Check the existing English emergency phrase list."""
    question_lower = normalize_text(question)

    for term in EMERGENCY_TERMS:
        if normalize_text(term) in question_lower:
            return True

    return False


def gemini_emergency_check(question):
    """Use Gemini to screen for a potential emergency in any language."""
    if use_mock or client is None:
        return False

    prompt = f"""
Read the patient's message below.

Decide whether it describes a POTENTIAL MEDICAL EMERGENCY that needs
urgent professional medical attention.

Consider the meaning of the message even if it is not written in English.

Return ONLY one word:
YES
or
NO

Patient message:
{question}
"""

    result = generate_text(
        prompt=prompt,
        system_instruction=(
            "You are a conservative medical-emergency screening classifier. "
            "Classify the message only. Do not diagnose the patient."
        )
    ).strip().upper()

    return result.startswith("YES")


def check_for_emergency(question):
    """Return True when the message may describe a medical emergency."""
    if not isinstance(question, str) or not question.strip():
        return False

    # Fast path for the large English keyword list.
    if english_emergency_match(question):
        return True

    # Multilingual semantic fallback.
    return gemini_emergency_check(question)


In [41]:
# Language detection

SUPPORTED_LANGUAGE_NAMES = [
    "English", "Vietnamese", "Spanish", "French", "German",
    "Greek", "Chinese", "Japanese", "Korean"
]


def detect_language(question):
    """Ask Gemini for the main language of the patient's message."""
    if not isinstance(question, str) or not question.strip():
        return "English"

    if use_mock or client is None:
        return "English"

    prompt = f"""
Identify the main language used in this patient message.

Return ONLY the language name. Do not explain.

Patient message:
{question}
"""

    result = generate_text(
        prompt=prompt,
        system_instruction="You are a language identification system."
    ).strip()

    result_lower = result.casefold()
    for language in SUPPORTED_LANGUAGE_NAMES:
        if language.casefold() in result_lower:
            return language

    return result.splitlines()[0].strip(" .,:;") or "English"


In [42]:
# Question intent classification

intent_categories = [
    "wound",
    "bleeding",
    "burn",
    "fracture_or_injury",
    "breathing",
    "allergic_reaction",
    "poisoning",
    "bite_or_sting",
    "eye_problem",
    "drowning",
    "seizure",
    "illness_or_symptoms",
    "first_aid_kit",
    "other"
]


def classify_intent(question):
    """Classify the main topic of the patient's question."""

    prompt = f"""
Classify the patient's medical question into ONE category.

Categories:
{", ".join(intent_categories)}

Return ONLY the category name.

Patient question:
{question}
"""

    result = generate_text(
        prompt=prompt,
        system_instruction=(
            "You are a medical question classification system. "
            "Classify the topic only. Do not diagnose."
        )
    ).strip().lower()

    for category in intent_categories:
        if category.lower() in result:
            return category

    return "other"

In [43]:
# Severity classification

severity_levels = [
    "low",
    "moderate",
    "high",
    "emergency"
]


def classify_severity(question):
    """Estimate the urgency level of the situation."""

    prompt = f"""
Classify the urgency of this health question.

Use exactly ONE of these levels:

LOW
MODERATE
HIGH
EMERGENCY

Definitions:

LOW:
Minor problem that can usually be handled with basic first aid.

MODERATE:
Needs attention but does not appear immediately life-threatening.

HIGH:
Potentially serious and should receive professional medical attention promptly.

EMERGENCY:
Could involve immediate danger and requires urgent professional help.

Do not diagnose.
Return ONLY the level.

Patient question:
{question}
"""

    result = generate_text(
        prompt=prompt,
        system_instruction=(
            "You are a conservative medical urgency classifier. "
            "Classify urgency only. Do not diagnose."
        )
    ).strip().lower()

    for level in severity_levels:
        if level in result:
            return level

    return "moderate"

In [44]:
system_instruction = """
Your name is MedBot, an educational first-aid and health-information AI assistant.

Your mission:
1. Briefly introduce yourself when appropriate.
2. Answer questions about health issues and first aid.
3. Recommend appropriate NON-MEDICINE items from the provided first-aid dataset.

Rules:
- Reply in the same language the patient uses, unless they explicitly request another language.
- Be calm, supportive, clear, and concise.
- Do not claim to diagnose a patient.
- Do not claim that information is "medically verified" unless it is explicitly provided as verified source material.
- Use the provided first-aid-kit information when recommending kit items.
- Never invent an item and say that it is in the kit.
- If the situation may be an emergency, encourage the patient to contact local emergency medical services or get help from a nearby trusted adult/person immediately.
- For emergencies, prioritize getting professional help over lengthy explanations.
- For vague questions, ask one or two important clarifying questions instead of repeatedly interrogating the patient.
- Explain important things the patient should avoid when relevant.
- If the question is unrelated to health or first aid, politely say that MedBot does not support that topic.
- Do not provide a diagnosis or pretend to replace a doctor or emergency professional.
"""

print(system_instruction)



Your name is MedBot, an educational first-aid and health-information AI assistant.

Your mission:
1. Briefly introduce yourself when appropriate.
2. Answer questions about health issues and first aid.
3. Recommend appropriate NON-MEDICINE items from the provided first-aid dataset.

Rules:
- Reply in the same language the patient uses, unless they explicitly request another language.
- Be calm, supportive, clear, and concise.
- Do not claim to diagnose a patient.
- Do not claim that information is "medically verified" unless it is explicitly provided as verified source material.
- Use the provided first-aid-kit information when recommending kit items.
- Never invent an item and say that it is in the kit.
- If the situation may be an emergency, encourage the patient to contact local emergency medical services or get help from a nearby trusted adult/person immediately.
- For emergencies, prioritize getting professional help over lengthy explanations.
- For vague questions, ask one or two

In [45]:
def emergency_response(question):
    """Generate a short emergency message in the patient's language."""
    language = detect_language(question)

    prompt = f"""
The patient's message may describe a medical emergency.

Write a SHORT, calm emergency response in {language}.

Requirements:
- Tell the person to contact their local emergency medical service or get
  immediate help from a nearby trusted adult/person.
- If they are with an injured person, tell them to follow instructions
  from emergency dispatchers or qualified medical professionals.
- Do not diagnose.
- Do not give a long list of treatment steps.
- Use the same language naturally and clearly.

Patient message:
{question}
"""

    return generate_text(
        prompt=prompt,
        system_instruction=(
            "You write concise, safety-focused emergency guidance. "
            "Do not diagnose or replace professional medical care."
        )
    )


def ask_bot(question):
    """Main MedBot pipeline."""

    if not isinstance(question, str) or not question.strip():
        return "Please enter a health or first-aid question."

    # ------------------------------------------------
    # 1. Emergency screening
    # ------------------------------------------------
    if check_for_emergency(question):
        return emergency_response(question)

    # ------------------------------------------------
    # 2. Analyze the question
    # ------------------------------------------------
    language = detect_language(question)
    intent = classify_intent(question)
    severity = classify_severity(question)

    # ------------------------------------------------
    # 3. Retrieve relevant first-aid information
    # ------------------------------------------------
    kit_context = get_kit_context(question)
    medical_context = get_medical_knowledge_context(question)

    # ------------------------------------------------
    # 4. Build a deeper Gemini prompt
    # ------------------------------------------------
    user_prompt = f"""
You are helping a patient with a health or first-aid question.

PATIENT LANGUAGE:
{language}

QUESTION CATEGORY:
{intent}

ESTIMATED URGENCY:
{severity}

RELEVANT MEDICAL KNOWLEDGE:
{medical_context}

RELEVANT FIRST-AID KIT INFORMATION:
{kit_context}

PATIENT'S QUESTION:
{question}

---

Instructions:

1. Answer in {language}, unless the patient explicitly asks for another language.
2. Do not diagnose the patient.
3. Explain the situation in a simple and understandable way.
4. Give practical first-aid information when appropriate.
5. Only recommend items that appear in the provided first-aid information.
6. Never invent a first-aid-kit item.
7. Clearly explain important things the patient should NOT do.
8. If professional medical care may be needed, say so clearly.
9. Do not overwhelm the patient with unnecessary information.
10. If the question is vague, ask one or two useful clarifying questions.
11. Do not pretend to know information that is not provided.
12. For potentially serious situations, prioritize safety over lengthy explanations.

Give the most useful answer you can based on the information provided.
"""

    response = generate_text(
        prompt=user_prompt,
        system_instruction=system_instruction,
    )

    return clean_response(response, question)

In [46]:
def clean_response(response, question):
    """Perform simple safety and quality checks on the generated response."""

    if not response or not response.strip():
        return "I was unable to generate a response. Please try again."

    # Prevent accidental claims of diagnosis.
    diagnosis_phrases = [
        "you definitely have",
        "you have been diagnosed with",
        "this confirms that you have"
    ]

    for phrase in diagnosis_phrases:
        if phrase in response.casefold():
            response = (
                "I cannot confirm a diagnosis. "
                + response
            )
            break

    return response.strip()

In [47]:
questions = [
    "What do I do if I get a snake bite?",
    "My arm is bleeding! What to do?!",
    "I can see somebody drowning, how do I deal with this?",
    "I have a very serious injury and I might die! Help me!",
    "What's the best burger spot in LA?",
    "I got a small scrape and it's bleeding a little bit, what to do?",
    "Tôi hiện tại đang chứng kiến một vụ xe tông và không biết có nên giúp hay không, phải làm gì?",
    "Βλέπω κάποιον να παθαίνει κρίση – τι πρέπει να κάνω; Παρακαλώ απαντήστε μου στα αγγλικά.",
    "¿Qué hago si tengo dificultad para respirar?",
]

for question in questions:
    print("Patient:", question)
    print("MedBot:", ask_bot(question))
    print("-" * 80)

Patient: What do I do if I get a snake bite?


MedBot: Call your local emergency medical services immediately or get help from a trusted adult nearby. 

If you or someone else is with an injured person, stay calm and follow all instructions from emergency dispatchers or medical professionals. Do not attempt to treat the bite yourself.
--------------------------------------------------------------------------------
Patient: My arm is bleeding! What to do?!
MedBot: Contact your local emergency medical service immediately, or get help from a nearby trusted person right away. 

If you are with an injured person, follow all instructions given by emergency dispatchers or medical professionals.
--------------------------------------------------------------------------------
Patient: I can see somebody drowning, how do I deal with this?
MedBot: Call your local emergency number (like 911) immediately and alert a lifeguard or nearby adult. 

If you are trained, safely assist from land or use a flotation device—do not put yourself in danger. 